# 🤖 AI Git Automation Agent - Graph Visualization

This notebook visualizes the LangGraph workflow for the AI Git Automation Agent.

## Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

## Agent State Definition

In [ ]:
class AgentState(TypedDict):
    has_changes: bool
    diff: str
    commit_message: str
    approved: bool
    committed: bool
    pushed: bool
    error: str

## Node Functions (Simplified)

In [ ]:
def detect_changes(state: AgentState) -> AgentState:
    print("🔍 Detecting changes...")
    state["has_changes"] = True
    return state

def analyze_diff(state: AgentState) -> AgentState:
    print("📊 Analyzing diff...")
    state["diff"] = "+ Added feature\n- Removed old code"
    return state

def generate_commit_message(state: AgentState) -> AgentState:
    print("🤖 Generating commit message...")
    state["commit_message"] = "feat: add new feature"
    return state

def human_approval(state: AgentState) -> AgentState:
    print("✋ Awaiting approval...")
    state["approved"] = True
    return state

def git_commit(state: AgentState) -> AgentState:
    print("💾 Committing...")
    state["committed"] = True
    return state

def git_push(state: AgentState) -> AgentState:
    print("🚀 Pushing...")
    state["pushed"] = True
    return state

## Build the Graph

In [ ]:
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("detect_changes", detect_changes)
workflow.add_node("analyze_diff", analyze_diff)
workflow.add_node("generate_commit_message", generate_commit_message)
workflow.add_node("human_approval", human_approval)
workflow.add_node("git_commit", git_commit)
workflow.add_node("git_push", git_push)

# Set entry point
workflow.set_entry_point("detect_changes")

# Add edges
workflow.add_conditional_edges(
    "detect_changes",
    lambda x: "analyze" if x["has_changes"] else "end",
    {"analyze": "analyze_diff", "end": END}
)

workflow.add_edge("analyze_diff", "generate_commit_message")
workflow.add_edge("generate_commit_message", "human_approval")

workflow.add_conditional_edges(
    "human_approval",
    lambda x: "commit" if x["approved"] else "end",
    {"commit": "git_commit", "end": END}
)

workflow.add_edge("git_commit", "git_push")
workflow.add_edge("git_push", END)

# Compile
graph = workflow.compile()
print("✅ Graph compiled!")

## Visualize the Graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"⚠️ Visualization needs: pip install pygraphviz")
    print(f"\nError: {e}")

## Text Visualization

In [ ]:
print("""
╔════════════════════════════════════════════════════════╗
║           AI GIT AUTOMATION WORKFLOW                   ║
╚════════════════════════════════════════════════════════╝

                    START
                      ↓
              🔍 detect_changes
                      ↓
            ┌─────────┴─────────┐
      has_changes?          no changes
            ↓                   ↓
      📊 analyze_diff          END
            ↓
   🤖 generate_commit_message
            ↓
      ✋ human_approval
            ↓
      ┌─────┴─────┐
  approved?    rejected
      ↓            ↓
 💾 git_commit    END
      ↓
 🚀 git_push
      ↓
     END
""")

## Run the Workflow

In [ ]:
initial_state = {
    "has_changes": False,
    "diff": "",
    "commit_message": "",
    "approved": False,
    "committed": False,
    "pushed": False,
    "error": ""
}

print("🚀 Running workflow...\n")
result = graph.invoke(initial_state)

print("\n📋 Final State:")
for key, value in result.items():
    print(f"  {key}: {value}")

## Mermaid Diagram

Copy this to https://mermaid.live to visualize:

In [ ]:
print("""
graph TD
    START([START]) --> A[🔍 detect_changes]
    A -->|has changes| B[📊 analyze_diff]
    A -->|no changes| END1([END])
    B --> C[🤖 generate_commit_message]
    C --> D[✋ human_approval]
    D -->|approved| E[💾 git_commit]
    D -->|rejected| END2([END])
    E --> F[🚀 git_push]
    F --> END3([END])
    
    style START fill:#90EE90
    style END1 fill:#FFB6C1
    style END2 fill:#FFB6C1
    style END3 fill:#90EE90
""")